# Notebook 8 – Feature Scaling
## Why Feature Scaling?

Look at our columns side by side:

* `Age`: roughly 18 to 999 (with errors still in there)
* `MonthlyIncome`: roughly 1,200 to 71,638
* `JobSatisfaction`: roughly 1 to 4

If you feed these into a model as-is, `MonthlyIncome` will completely
dominate any distance-based calculation, not because it's more important,
but purely because its numbers are bigger. Scaling puts every column on a
comparable footing so the model judges features by their actual signal,
not their unit size.

In [9]:
import pandas as pd

df = pd.read_csv("hr_employee_attrition_raw.csv")
df["MonthlyIncome_clean"] = pd.to_numeric(
    df["MonthlyIncome"].astype(str).str.replace(" USD", "", regex=False).str.strip(),
    errors="coerce"
)
df[["Age", "MonthlyIncome_clean", "DistanceFromHomeKM", "JobSatisfaction"]].describe()

,Age,MonthlyIncome_clean,DistanceFromHomeKM,JobSatisfaction
count,1230.000000,1169.000000,1206.000000,1179.000000
mean,36.600000,5785.085765,11.067696,2.513995
std,41.211921,4138.929438,46.222238,1.124854
min,18.000000,1200.000000,0.500000,1.000000
25%,28.000000,4023.490000,2.225000,1.000000
50%,34.000000,5496.300000,5.300000,3.000000
75%,40.000000,6927.260000,10.900000,4.000000
max,999.000000,71638.178357,830.316096,4.000000


## Scale-Sensitive vs Scale-Insensitive Algorithms

Not every algorithm needs scaling, so it's worth knowing which is which
before doing it automatically.

**Scale-sensitive** (scaling matters a lot):
* KNN, K-Means (distance-based)
* Linear/Logistic Regression, SVM (gradient-based, or margin-based)
* Neural Networks (gradient descent converges much slower/worse without it)
* PCA (variance-based, dominated by high-magnitude columns otherwise)

**Scale-insensitive** (scaling usually not needed):
* Decision Trees, Random Forest, Gradient Boosting (XGBoost, LightGBM)

These split data based on threshold comparisons on one feature at a time,
so whether `MonthlyIncome` is in the thousands or scaled to 0 to 1 makes no
difference to how the tree splits.

## Normalization vs Standardization

These two terms get used interchangeably, but they're different operations.

* **Normalization** rescales values into a fixed range, usually 0 to 1
  (this is what Min-Max Scaling does).
* **Standardization** rescales values to have a mean of 0 and a standard
  deviation of 1, without bounding them to a fixed range.

Neither is "better" universally, it depends on the algorithm and whether
your data has outliers.

## StandardScaler (Standardization)

**Formula:** `(x - mean) / std`

**Best for:** algorithms that assume roughly normally distributed data
(Linear Regression, Logistic Regression, PCA, SVM).

**How outliers affect it:** badly. The `mean` and `std` are both calculated
using every value, including outliers. Our `Age` column has values up to
999, so the mean (36.6) and std (41.2) are both dragged upward by a handful
of broken rows. Every normal-range age (like 30 or 45) then gets scaled
using a distorted reference point, compressing the real, useful variation
into a tiny sliver of the scaled range.

In [10]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
df["Age_standard_scaled"] = scaler.fit_transform(df[["Age"]])

print("Mean used for scaling:", scaler.mean_[0].round(2))
print("Std used for scaling:", scaler.scale_[0].round(2))
df[["Age", "Age_standard_scaled"]].sort_values("Age", ascending=False).head()

Mean used for scaling: 36.6
Std used for scaling: 41.2


,Age,Age_standard_scaled
1199,999,23.361965
636,999,23.361965
318,200,3.966485
772,200,3.966485
1137,200,3.966485


## MinMaxScaler (Normalization)

**Formula:** `(x - min) / (max - min)`, squeezing everything into 0 to 1.

**Best for:** neural networks, image data, or algorithms that need a fixed
bounded range.

**How outliers affect it:** even worse than StandardScaler, because a
single extreme value directly sets one end of the range. `DistanceFromHomeKM`
has a max of 830 km (almost certainly a data error), which means every
realistic commute distance (5 km, 20 km, even 60 km) gets squeezed into
values under 0.1, right next to zero, because the scale is entirely
anchored to one broken row.

In [11]:
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
df["Distance_minmax_scaled"] = scaler.fit_transform(df[["DistanceFromHomeKM"]])

# See how badly one outlier compresses everything else
df[["DistanceFromHomeKM", "Distance_minmax_scaled"]].sort_values("DistanceFromHomeKM").head(10)

,DistanceFromHomeKM,Distance_minmax_scaled
915,0.5,0.0
914,0.5,0.0
350,0.5,0.0
62,0.5,0.0
896,0.5,0.0
962,0.5,0.0
374,0.5,0.0
889,0.5,0.0
884,0.5,0.0
1120,0.5,0.0


## RobustScaler

**Formula:** `(x - median) / IQR`, using the median and interquartile range
instead of mean and min/max.

**Best for:** any column where outliers are present but you don't want to
remove them first (exactly our situation with `Age`, `MonthlyIncome`, and
`DistanceFromHomeKM`).

**How outliers affect it:** far less than the other two. Since the median
and IQR are calculated from the middle 50% of the data, a handful of extreme
values (like Age = 999) barely move the reference points at all. This is
usually the safer default when you know a column has outliers but haven't
fully cleaned them yet.

In [12]:
from sklearn.preprocessing import RobustScaler

scaler = RobustScaler()
df["Distance_robust_scaled"] = scaler.fit_transform(df[["DistanceFromHomeKM"]])

# Compare: MinMax squeezed everything near 0, RobustScaler keeps real variation visible
df[["DistanceFromHomeKM", "Distance_minmax_scaled", "Distance_robust_scaled"]].sort_values(
    "DistanceFromHomeKM"
).head(10)

,DistanceFromHomeKM,Distance_minmax_scaled,Distance_robust_scaled
915,0.5,0.0,-0.553314
914,0.5,0.0,-0.553314
350,0.5,0.0,-0.553314
62,0.5,0.0,-0.553314
896,0.5,0.0,-0.553314
962,0.5,0.0,-0.553314
374,0.5,0.0,-0.553314
889,0.5,0.0,-0.553314
884,0.5,0.0,-0.553314
1120,0.5,0.0,-0.553314


## MaxAbsScaler

**Formula:** `x / max(abs(x))`, scales by the maximum absolute value only,
keeping 0 fixed at 0.

**Best for:** sparse data (lots of zeros), like one-hot encoded columns or
frequency-encoded features, where you don't want to shift the zero point.

**How outliers affect it:** same core problem as MinMaxScaler, since it's
also anchored entirely to the single largest value in the column. One
outlier still compresses everything else toward zero.

In [13]:
from sklearn.preprocessing import MaxAbsScaler

scaler = MaxAbsScaler()
df["Income_maxabs_scaled"] = scaler.fit_transform(df[["MonthlyIncome_clean"]])
df[["MonthlyIncome_clean", "Income_maxabs_scaled"]].sort_values(
    "MonthlyIncome_clean", ascending=False
).head()

,MonthlyIncome_clean,Income_maxabs_scaled
881,71638.178357,1.000000
562,55866.897265,0.779848
419,48267.700403,0.673771
1054,43532.998753,0.607679
341,41493.123557,0.579204


## Side-by-Side Comparison

* **StandardScaler**: uses mean and std. Range is unbounded. Outlier
  sensitivity is high, outliers distort both the mean and the std.
* **MinMaxScaler**: uses min and max. Range is fixed at 0 to 1. Outlier
  sensitivity is very high, one outlier sets the entire scale.
* **RobustScaler**: uses median and IQR. Range is unbounded. Outlier
  sensitivity is low, it's resistant by design.
* **MaxAbsScaler**: uses the max absolute value. Range is -1 to 1. Outlier
  sensitivity is high, similar issue to MinMax.

In [14]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler, RobustScaler, MaxAbsScaler

comparison = df[["DistanceFromHomeKM"]].copy()
comparison["Standard"] = StandardScaler().fit_transform(df[["DistanceFromHomeKM"]])
comparison["MinMax"] = MinMaxScaler().fit_transform(df[["DistanceFromHomeKM"]])
comparison["Robust"] = RobustScaler().fit_transform(df[["DistanceFromHomeKM"]])
comparison["MaxAbs"] = MaxAbsScaler().fit_transform(df[["DistanceFromHomeKM"]])

comparison.sort_values("DistanceFromHomeKM").head(10)

,DistanceFromHomeKM,Standard,MinMax,Robust,MaxAbs
915,0.5,-0.228723,0.0,-0.553314,0.000602
914,0.5,-0.228723,0.0,-0.553314,0.000602
350,0.5,-0.228723,0.0,-0.553314,0.000602
62,0.5,-0.228723,0.0,-0.553314,0.000602
896,0.5,-0.228723,0.0,-0.553314,0.000602
962,0.5,-0.228723,0.0,-0.553314,0.000602
374,0.5,-0.228723,0.0,-0.553314,0.000602
889,0.5,-0.228723,0.0,-0.553314,0.000602
884,0.5,-0.228723,0.0,-0.553314,0.000602
1120,0.5,-0.228723,0.0,-0.553314,0.000602
